# Chapter 23 — Log Collection and Processing
## Turning Millions of Syslog Lines into Operational Intelligence

Every network device is continuously narrating its own story.
OSPF adjacency changes, BGP prefix withdrawals, authentication failures,
interface flaps — all of it ends up in syslog.

The problem: most organizations collect logs the way people collect old newspapers.
They put them somewhere, they don't throw them away, and they almost never read them
**until something has already gone catastrophically wrong.**

AI changes this. The bottleneck in network operations is not lack of data —
it is the inability to process the volume that devices generate.

```
Medium enterprise: millions of syslog messages per day
Human team: reads hundreds per shift (if they read them at all)
AI pipeline: reads ALL of them, every second
```

The five-stage log pipeline — and the AI techniques that make it work:

| Stage | What Happens | AI Role |
|-------|-------------|---------|
| **1 — Generate** | Device creates syslog message | (source) |
| **2 — Transport** | UDP/TCP syslog to collector | Loss detection |
| **3 — Parse** | Extract structured fields from free text | Regex + LLM |
| **4 — Enrich** | Add CMDB, topology, change context | RAG lookup |
| **5 — Analyze** | Anomaly detection, alerting | LLM reasoning |

> **No extra libraries needed.** All demos use Python built-ins + the Anthropic API.

### Install
```
pip install anthropic
```


## Setup — API Client and Sample Log Data

In [ ]:
import json, re, time
from anthropic import Anthropic
from collections import defaultdict, Counter

# ── API key setup (Colab-safe with getpass fallback) ─────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=api_key)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system="You are a network operations AI assistant."):
    r = client.messages.create(
        model=model, max_tokens=1024, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Sample raw syslog messages (realistic Cisco IOS format) ─────────────────────
# Format: <PRI>Mmm DD HH:MM:SS hostname %FACILITY-SEVERITY-MNEMONIC: message
RAW_SYSLOGS = [
    "<189>Feb 25 02:14:01 R1 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.12.2 on Gi0/0 from LOADING to FULL, Loading Done",
    "<189>Feb 25 02:14:05 R2 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.12.1 on Gi0/0 from FULL to EXSTART, Interface state change",
    "<189>Feb 25 02:14:07 R2 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.12.1 on Gi0/0 from EXSTART to FULL, Loading Done",
    "<189>Feb 25 02:15:00 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet1/0/24, changed state to down",
    "<189>Feb 25 02:15:01 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet1/0/24, changed state to up",
    "<185>Feb 25 02:15:02 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet1/0/24, changed state to down",
    "<185>Feb 25 02:15:03 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet1/0/24, changed state to up",
    "<177>Feb 25 02:16:10 R3 %BGP-3-NOTIFICATION: sent to neighbor 203.0.113.1 active 4/0 (hold time expired)",
    "<177>Feb 25 02:16:12 R3 %BGP-5-ADJCHANGE: neighbor 203.0.113.1 Down BGP Notification sent",
    "<189>Feb 25 02:17:00 FW1 %SEC-6-IPACCESSLOGP: list BLOCK-IN denied tcp 198.51.100.4(54321) -> 10.0.1.100(22), 47 packets",
    "<189>Feb 25 02:17:01 FW1 %SEC-6-IPACCESSLOGP: list BLOCK-IN denied tcp 198.51.100.4(54322) -> 10.0.1.100(22), 1 packets",
    "<189>Feb 25 02:17:02 FW1 %SEC-6-IPACCESSLOGP: list BLOCK-IN denied tcp 198.51.100.4(54323) -> 10.0.1.100(22), 1 packets",
    "<161>Feb 25 02:18:00 R1 %LINK-3-UPDOWN: Interface GigabitEthernet0/1, changed state to down",
    "<161>Feb 25 02:18:05 R1 %LINK-5-CHANGED: Interface GigabitEthernet0/1, changed state to administratively down",
    # Novel/unparsed format (not matching standard regex)
    "Feb 25 02:19:00 NXOS-LEAF1 ETHPM-1-IF_DOWN_LINK_FAILURE: Interface Ethernet1/47 is down (Link failure)",
]

print(f"Sample log dataset: {len(RAW_SYSLOGS)} messages")
for i, msg in enumerate(RAW_SYSLOGS[:5], 1):
    print(f"  {i}. {msg[:80]}...")
print(f"  ... and {len(RAW_SYSLOGS)-5} more")


---
## Demo 1 — Regex Log Parser: Structured Fields from Free Text

Raw syslog is free-form text. Before AI can reason about it, we need
**structured fields** — timestamp, device, facility, severity, mnemonic, key values.

```
Raw:    "<189>Feb 25 02:14:01 R1 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.12.2 on Gi0/0 from LOADING to FULL"

Parsed: {
  "device":    "R1",
  "facility":  "OSPF",
  "severity":  5,         ← Notification (0=Emergency, 7=Debug)
  "mnemonic":  "ADJCHG",
  "neighbor":  "10.0.12.2",
  "interface": "Gi0/0",
  "old_state": "LOADING",
  "new_state": "FULL"
}
```

The regex approach works well for **high-volume common message types**
(OSPF events, interface state changes, BGP events — millions per day).
For rare or novel message types, we use the LLM (Demo 2).


In [ ]:
# ── Regex-Based Syslog Parser ─────────────────────────────────────────────────

# Cisco IOS severity names
SEVERITY_NAMES = {
    0: "Emergency", 1: "Alert", 2: "Critical", 3: "Error",
    4: "Warning", 5: "Notification", 6: "Informational", 7: "Debug"
}

# ── Base syslog structure parser ──────────────────────────────────────────────
SYSLOG_BASE = re.compile(
    r'(?:<(?P<pri>\d+)>)?'
    r'(?P<month>\w+)\s+(?P<day>\d+)\s+(?P<time>\d+:\d+:\d+)\s+'
    r'(?P<device>\S+)\s+'
    r'%(?P<facility>[A-Z0-9_]+)-(?P<severity>\d)-(?P<mnemonic>[A-Z0-9_]+):\s*'
    r'(?P<message>.+)'
)

# ── Event-specific field extractors ──────────────────────────────────────────
OSPF_ADJCHG = re.compile(
    r'Process\s+(?P<process>\d+),\s+Nbr\s+(?P<neighbor>[\d.]+)\s+'
    r'on\s+(?P<interface>\S+)\s+from\s+(?P<old_state>\w+)\s+to\s+(?P<new_state>\w+)'
)

LINK_UPDOWN = re.compile(
    r'Interface\s+(?P<interface>\S+),\s+changed\s+state\s+to\s+(?P<state>\w.*)'
)

BGP_ADJCHANGE = re.compile(
    r'neighbor\s+(?P<neighbor>[\d.]+)\s+(?P<direction>\w+)\s+(?P<reason>.+)'
)

ACL_DENY = re.compile(
    r'list\s+(?P<acl>\S+)\s+denied\s+(?P<proto>\w+)\s+'
    r'(?P<src_ip>[\d.]+)\((?P<src_port>\d+)\)\s*->\s*'
    r'(?P<dst_ip>[\d.]+)\((?P<dst_port>\d+)\),\s+(?P<count>\d+)\s+packets'
)

EVENT_PARSERS = {
    "ADJCHG":     OSPF_ADJCHG,
    "UPDOWN":     LINK_UPDOWN,
    "CHANGED":    LINK_UPDOWN,
    "ADJCHANGE":  BGP_ADJCHANGE,
    "IPACCESSLOGP": ACL_DENY,
}


def parse_syslog(raw: str) -> dict | None:
    """Parse a raw syslog line into a structured dict. Returns None if unrecognized."""
    m = SYSLOG_BASE.match(raw.strip())
    if not m:
        return None

    result = {
        "timestamp": f"{m.group('month')} {m.group('day')} {m.group('time')}",
        "device":    m.group("device"),
        "facility":  m.group("facility"),
        "severity":  int(m.group("severity")),
        "severity_name": SEVERITY_NAMES.get(int(m.group("severity")), "Unknown"),
        "mnemonic":  m.group("mnemonic"),
        "raw_message": m.group("message"),
        "event_type": None,
        "fields": {}
    }

    # Try event-specific extraction
    mnemonic = m.group("mnemonic")
    if mnemonic in EVENT_PARSERS:
        em = EVENT_PARSERS[mnemonic].search(m.group("message"))
        if em:
            result["event_type"] = mnemonic
            result["fields"] = em.groupdict()

    return result


# Parse all sample logs
parsed = []
unparsed = []
for raw in RAW_SYSLOGS:
    p = parse_syslog(raw)
    if p:
        parsed.append(p)
    else:
        unparsed.append(raw)

print(f"Parsed:   {len(parsed)} / {len(RAW_SYSLOGS)} messages ({len(parsed)/len(RAW_SYSLOGS)*100:.0f}%)")
print(f"Unparsed: {len(unparsed)} messages (→ will go to LLM parser in Demo 2)")
print()

# Display structured output
for p in parsed[:4]:
    print(f"[{p['timestamp']}] {p['device']} | {p['facility']}-{p['severity']}-{p['mnemonic']}")
    print(f"  Severity: {p['severity_name']}")
    if p["fields"]:
        for k, v in p["fields"].items():
            print(f"  {k}: {v}")
    print()


---
## Demo 2 — LLM Log Parser: Handle What Regex Can't

Regex breaks when:
- Message format varies across OS versions
- The message is from a vendor the regex library doesn't cover (NX-OS, JunOS, etc.)
- A new event type appears that no pattern was written for

**LLM-based parsing** understands the *semantics* of the message, not just the pattern.
It handles variations and novel message types naturally.

The tradeoff: LLM parsing is ~100x slower and more expensive than regex.

**Practical architecture**: Regex for the common 90% (millions/day), LLM for the rare 10% (novel, complex, or multi-vendor messages). This is the **hybrid parsing** approach from the chapter.

```
Raw log →  [Regex parser]  →  ✓ parsed (fast, cheap)
                │
                └─ ✗ no match →  [LLM parser]  →  ✓ parsed (slow, smart)
```


In [ ]:
# ── LLM-Based Log Parser for Novel/Unrecognized Messages ─────────────────────

def llm_parse_log(raw: str) -> dict:
    """
    Uses Haiku to parse any log message into structured JSON.
    Handles NX-OS, JunOS, novel event types — anything regex misses.
    """
    prompt = f"""Parse this network device log message into structured JSON.

Raw message: {raw}

Extract ALL useful fields. Common fields to look for:
- timestamp, device, facility, severity (0-7), mnemonic
- interface name, IP address, protocol, state changes
- error codes, counts, descriptions

Return ONLY valid JSON:
{{
  "timestamp": "...",
  "device": "...",
  "facility": "...",
  "severity": 5,
  "severity_name": "Notification",
  "mnemonic": "...",
  "event_type": "...",
  "fields": {{
    "interface": "...",
    "state": "..."
  }},
  "raw_message": "...",
  "parse_method": "llm"
}}
"""
    raw_response = ask(prompt, model=HAIKU,
                       system="You are a log parsing engine. Return only valid JSON.")
    match = re.search(r'\{.*\}', raw_response, re.DOTALL)
    if match:
        try:
            result = json.loads(match.group())
            result["parse_method"] = "llm"
            return result
        except Exception:
            pass
    return {"raw_message": raw, "parse_method": "llm_failed", "fields": {}}


def hybrid_parse_all(raw_logs: list[str]) -> list[dict]:
    """
    Hybrid pipeline: try regex first, fall back to LLM for unrecognized messages.
    """
    results = []
    llm_count = 0
    regex_count = 0

    for raw in raw_logs:
        p = parse_syslog(raw)
        if p:
            p["parse_method"] = "regex"
            results.append(p)
            regex_count += 1
        else:
            # Fall back to LLM
            p = llm_parse_log(raw)
            results.append(p)
            llm_count += 1

    print(f"Parsing complete:")
    print(f"  Regex-parsed : {regex_count} messages  (fast, cheap)")
    print(f"  LLM-parsed   : {llm_count} messages    (smart, handles novel formats)")
    print(f"  Total        : {len(results)} messages")
    return results


all_parsed = hybrid_parse_all(RAW_SYSLOGS)

print()
print("LLM-parsed messages:")
for p in all_parsed:
    if p.get("parse_method") == "llm":
        print(f"\n  Raw: {p.get('raw_message', '')[:80]}")
        print(f"  Device: {p.get('device', 'unknown')}")
        print(f"  Facility: {p.get('facility', 'unknown')}")
        print(f"  Fields: {p.get('fields', {})}")
        print(f"  Event type: {p.get('event_type', 'unknown')}")


---
## Demo 3 — Log Enrichment: Add Context That Changes Everything

A raw log message tells you *what* happened on the device.
Enrichment tells you *why it matters* by adding external context:

```
Raw:      "authentication failure from 192.168.1.5"

Enriched: "authentication failure from 192.168.1.5
           → Host: Finance-WS-042 (Finance Dept)
           → User: jsmith (standard user, no admin rights)
           → Site: Budapest HQ
           → Change window: None active
           → Count: 47th failure in 10 minutes   ← now this is an alert!"
```

Three enrichment sources:
1. **CMDB** — maps device/IP to hostname, site, role, owner
2. **Topology context** — maps interface to connected peer, circuit ID
3. **Change management** — was a maintenance window active? Was a change ticket open?


In [ ]:
# ── Log Enrichment Pipeline ───────────────────────────────────────────────────

# Simulated CMDB (Configuration Management Database)
CMDB = {
    "R1":         {"hostname": "R1-CORE-HQ",   "site": "Budapest HQ", "role": "Core Router",     "owner": "netops"},
    "R2":         {"hostname": "R2-DIST-HQ",   "site": "Budapest HQ", "role": "Distribution",    "owner": "netops"},
    "R3":         {"hostname": "R3-EDGE",       "site": "Budapest HQ", "role": "Edge Router",     "owner": "netops"},
    "CORE-SW1":   {"hostname": "SW1-CORE-HQ",  "site": "Budapest HQ", "role": "Core Switch",     "owner": "netops"},
    "FW1":        {"hostname": "FW1-EDGE",      "site": "Budapest HQ", "role": "Firewall",        "owner": "security"},
    "NXOS-LEAF1": {"hostname": "LEAF1-DC",      "site": "Data Center", "role": "Leaf Switch",     "owner": "dc-ops"},
}

# Simulated topology — maps device+interface to peer
TOPOLOGY = {
    ("R1", "Gi0/0"):              {"peer": "R2", "peer_intf": "Gi0/0", "circuit": "ILL-R1-R2-001"},
    ("R1", "Gi0/1"):              {"peer": "R3", "peer_intf": "Gi0/1", "circuit": "ILL-R1-R3-001"},
    ("R2", "Gi0/0"):              {"peer": "R1", "peer_intf": "Gi0/0", "circuit": "ILL-R1-R2-001"},
    ("CORE-SW1", "Gi1/0/24"):     {"peer": "R1",  "peer_intf": "Gi0/2", "circuit": "AGG-SW1-R1-001"},
    ("NXOS-LEAF1", "Ethernet1/47"):{"peer": "SPINE1", "peer_intf": "Eth1/1", "circuit": "FABRIC-LEAF1-SPINE1"},
}

# Simulated change windows
CHANGE_WINDOWS = [
    {"id": "CHG0045832", "start": "02:00", "end": "04:00",
     "devices": ["CORE-SW1", "R1"], "description": "Core switch firmware upgrade"},
]

# Simulated IP-to-host mapping (for security events)
IP_HOSTS = {
    "198.51.100.4": {"hostname": "UNKNOWN-EXTERNAL", "type": "external", "threat_intel": "known-scanner"},
    "10.0.1.100":   {"hostname": "SRV-MGMT-01",      "type": "server",   "role": "management"},
    "192.168.1.5":  {"hostname": "Finance-WS-042",   "type": "workstation", "department": "Finance"},
}


def enrich_log(parsed: dict) -> dict:
    """Add CMDB, topology, change context to a parsed log event."""
    enriched = dict(parsed)
    enriched["enrichment"] = {}
    device = parsed.get("device", "")

    # ── CMDB enrichment ───────────────────────────────────────────────────────
    if device in CMDB:
        enriched["enrichment"]["cmdb"] = CMDB[device]

    # ── Topology enrichment ───────────────────────────────────────────────────
    interface = parsed.get("fields", {}).get("interface", "")
    if interface and device:
        topo_key = (device, interface)
        if topo_key in TOPOLOGY:
            enriched["enrichment"]["topology"] = TOPOLOGY[topo_key]

    # ── Change window enrichment ──────────────────────────────────────────────
    for cw in CHANGE_WINDOWS:
        if device in cw["devices"]:
            enriched["enrichment"]["change_window"] = {
                "active": True,
                "change_id": cw["id"],
                "description": cw["description"],
            }
            break

    # ── Security enrichment (IP addresses in ACL denies) ─────────────────────
    if parsed.get("mnemonic") == "IPACCESSLOGP":
        src_ip = parsed.get("fields", {}).get("src_ip", "")
        dst_ip = parsed.get("fields", {}).get("dst_ip", "")
        if src_ip in IP_HOSTS:
            enriched["enrichment"]["src_host"] = IP_HOSTS[src_ip]
        if dst_ip in IP_HOSTS:
            enriched["enrichment"]["dst_host"] = IP_HOSTS[dst_ip]

    return enriched


# Enrich all parsed logs
enriched_logs = [enrich_log(p) for p in all_parsed]

# Display enriched events
print("Enriched log examples:")
print("=" * 60)
for ev in enriched_logs:
    if ev.get("enrichment"):
        print(f"\n[{ev.get('timestamp','')}] {ev.get('device','')} | {ev.get('facility','')}-{ev.get('mnemonic','')}")
        print(f"  Raw: {ev.get('raw_message','')[:60]}...")
        for key, val in ev["enrichment"].items():
            print(f"  Enrichment [{key}]: {val}")


---
## Demo 4 — Anomaly Detection: Rate, Pattern, and AI Analysis

Once logs are parsed and enriched, AI can find what humans miss:

**Rate-based anomalies**: An interface that toggles 3× is a flap.
One that toggles 300× in a minute is a broadcast storm. Same event type — very different severity.

**Pattern-based anomalies**: Three SSH failures from the same external IP across
47 packets is not "informational" — it's a brute-force attempt.

**Correlation anomalies**: BGP session goes down 30 seconds after a firewall ACL deny.
No regex catches this cross-device correlation.

```
Enriched log stream
        │
   [Rate counter]     → interface toggled 48× in 60s → HIGH SEVERITY
        │
   [Pattern detector] → 3 ACL denies same src IP → SECURITY ALERT
        │
   [LLM analyzer]     → correlates events across devices → root cause hypothesis
```


In [ ]:
# ── Anomaly Detection Engine ──────────────────────────────────────────────────

def detect_rate_anomalies(logs: list[dict]) -> list[dict]:
    """
    Detect events that occur at abnormal rates.
    Interface flapping, rapid ACL denies, repeated auth failures.
    """
    anomalies = []

    # Count events per (device, interface/key) in the time window
    interface_events = defaultdict(list)
    acl_deny_src     = defaultdict(int)

    for ev in logs:
        mnemonic = ev.get("mnemonic", "")
        device   = ev.get("device", "")
        fields   = ev.get("fields", {})

        # Track interface state changes
        if mnemonic in ("UPDOWN", "CHANGED") and fields.get("interface"):
            key = f"{device}:{fields['interface']}"
            interface_events[key].append(ev)

        # Track ACL denies by source IP
        if mnemonic == "IPACCESSLOGP":
            src = fields.get("src_ip", "unknown")
            acl_deny_src[src] += int(fields.get("count", 1))

    # Flag interfaces with too many state changes
    for key, events in interface_events.items():
        count = len(events)
        if count >= 2:
            device, interface = key.split(":", 1)
            severity = "CRITICAL" if count >= 10 else "HIGH" if count >= 4 else "WARNING"
            anomalies.append({
                "type": "INTERFACE_FLAP",
                "device": device,
                "interface": interface,
                "event_count": count,
                "severity": severity,
                "description": f"Interface {interface} on {device} changed state {count}× — possible physical issue",
                "enrichment": events[0].get("enrichment", {}),
            })

    # Flag source IPs with high ACL deny counts
    for src_ip, count in acl_deny_src.items():
        if count >= 3:
            anomalies.append({
                "type": "SECURITY_SCAN",
                "src_ip": src_ip,
                "denied_packets": count,
                "severity": "HIGH" if count >= 40 else "WARNING",
                "description": f"Source {src_ip} has {count} ACL-denied packets — possible port scan or brute-force",
            })

    return anomalies


def llm_analyze_anomalies(anomalies: list[dict], enriched_logs: list[dict]) -> str:
    """
    Use Sonnet to correlate anomalies and suggest root causes.
    This is the AI layer that catches patterns no regex can find.
    """
    # Build a summary of the log window for context
    event_summary = []
    for ev in enriched_logs:
        event_summary.append({
            "time": ev.get("timestamp", ""),
            "device": ev.get("device", ""),
            "event": f"{ev.get('facility','')}-{ev.get('mnemonic','')}",
            "details": ev.get("raw_message", "")[:80],
            "context": ev.get("enrichment", {}),
        })

    prompt = f"""You are a NOC analyst reviewing network anomalies.

Anomalies detected in the last 5 minutes:
{json.dumps(anomalies, indent=2)}

Full event timeline (for correlation):
{json.dumps(event_summary[:10], indent=2)}

Analyze:
1. Which anomalies are related to each other? (cross-device correlation)
2. What is the most likely root cause for each group?
3. What is the urgency (P1/P2/P3)?
4. What is the recommended first action?

Be specific. Mention device names, interface names, and timestamps.
"""
    return ask(prompt, model=SONNET,
               system="You are a senior NOC analyst. Provide precise, actionable analysis.")


# Run anomaly detection
anomalies = detect_rate_anomalies(enriched_logs)

print(f"Anomalies detected: {len(anomalies)}")
print("=" * 60)
for a in anomalies:
    sev_icon = {"CRITICAL": "🚨", "HIGH": "🔴", "WARNING": "⚠"}.get(a["severity"], "?")
    print(f"\n{sev_icon} [{a['severity']}] {a['type']}")
    print(f"  {a['description']}")
    if a.get("enrichment"):
        cmdb = a["enrichment"].get("cmdb", {})
        if cmdb:
            print(f"  Device context: {cmdb.get('role','?')} at {cmdb.get('site','?')}")
        if a["enrichment"].get("change_window", {}).get("active"):
            cw = a["enrichment"]["change_window"]
            print(f"  ⚠ Active change window: {cw['change_id']} — {cw['description']}")

# LLM correlation analysis
if anomalies:
    print("\n" + "=" * 60)
    print("LLM Correlation Analysis:")
    print("=" * 60)
    analysis = llm_analyze_anomalies(anomalies, enriched_logs)
    print(analysis)


---
## Demo 5 — Full Log Intelligence Pipeline

The complete pipeline from raw syslog to actionable alert:

```
Raw syslog stream (millions/day)
        │
   [Stage 3: Hybrid Parser]  → Regex for common types, LLM for novel
        │
   [Stage 4: Enrichment]     → CMDB + topology + change window context
        │
   [Stage 5: Anomaly Engine] → Rate detection + pattern matching
        │
   [LLM Correlation]         → Cross-device root cause analysis
        │
   [Alert Generator]         → Priority-ranked, contextual NOC alert
```

This is the AI-native alternative to reading millions of syslog lines by hand.
Every message is processed, every anomaly is detected, every correlation is found.


In [ ]:
# ── Full Log Intelligence Pipeline ───────────────────────────────────────────

def generate_noc_alert(anomalies: list[dict], analysis: str,
                       total_logs: int, parse_stats: dict) -> str:
    """Auto-generate a structured NOC alert from the analysis."""
    critical = [a for a in anomalies if a["severity"] == "CRITICAL"]
    high     = [a for a in anomalies if a["severity"] == "HIGH"]

    prompt = f"""Generate a concise NOC alert report.

Pipeline stats: {total_logs} logs processed in this window
Parse stats: {parse_stats}
Critical anomalies: {len(critical)}
High anomalies: {len(high)}

AI correlation analysis:
{analysis}

Format as:
NOC ALERT — [timestamp]
PRIORITY: P1/P2/P3
SUMMARY: (one sentence)
AFFECTED: (devices/services)
ROOT CAUSE: (specific)
IMMEDIATE ACTION:
  1. ...
  2. ...
RELATED CHANGE: (change ticket if applicable, else None)
AUTO-RESOLVED: Yes/No
"""
    return ask(prompt, model=HAIKU,
               system="You are a NOC alert system. Write clear, actionable alerts.")


def run_full_pipeline(raw_logs: list[str]):
    print("LOG INTELLIGENCE PIPELINE")
    print("=" * 70)
    start = time.time()

    # ── Stage 1: Hybrid parsing ───────────────────────────────────────────────
    print(f"\n[1/4] Hybrid parsing {len(raw_logs)} raw log messages...")
    parsed_logs = hybrid_parse_all(raw_logs)

    regex_count = sum(1 for p in parsed_logs if p.get("parse_method") == "regex")
    llm_count   = sum(1 for p in parsed_logs if p.get("parse_method") == "llm")
    parse_stats = {"regex": regex_count, "llm": llm_count, "failed": len(raw_logs) - regex_count - llm_count}

    # ── Stage 2: Enrichment ───────────────────────────────────────────────────
    print(f"\n[2/4] Enriching {len(parsed_logs)} parsed events...")
    enriched = [enrich_log(p) for p in parsed_logs]
    enriched_count = sum(1 for e in enriched if e.get("enrichment"))
    print(f"  ✓ {enriched_count} events enriched with CMDB/topology/change context")

    # ── Stage 3: Anomaly detection ────────────────────────────────────────────
    print("\n[3/4] Running anomaly detection...")
    anomalies = detect_rate_anomalies(enriched)
    severity_counts = Counter(a["severity"] for a in anomalies)
    for sev, count in sorted(severity_counts.items()):
        icon = {"CRITICAL": "🚨", "HIGH": "🔴", "WARNING": "⚠"}.get(sev, "?")
        print(f"  {icon} {sev}: {count} anomaly(ies)")

    # ── Stage 4: LLM correlation + alert ─────────────────────────────────────
    print("\n[4/4] LLM correlation analysis and alert generation...")
    if anomalies:
        analysis  = llm_analyze_anomalies(anomalies, enriched)
        noc_alert = generate_noc_alert(anomalies, analysis, len(raw_logs), parse_stats)
    else:
        analysis  = "No anomalies detected — network operating normally."
        noc_alert = "NOC ALERT — No issues. All systems nominal."

    elapsed = time.time() - start

    # ── Final output ──────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("GENERATED NOC ALERT")
    print("=" * 70)
    print(noc_alert)
    print("\n" + "=" * 70)
    print(f"Pipeline summary:")
    print(f"  Logs processed : {len(raw_logs)}")
    print(f"  Parse method   : {regex_count} regex / {llm_count} LLM")
    print(f"  Events enriched: {enriched_count}")
    print(f"  Anomalies found: {len(anomalies)}")
    print(f"  Pipeline time  : {elapsed:.1f}s")
    print("=" * 70)


run_full_pipeline(RAW_SYSLOGS)


---
## Summary — From Log Noise to Operational Intelligence

### The Five-Stage Pipeline

```
Stage 1: Generate  → Device writes syslog (first signal loss point: buffer overflow)
Stage 2: Transport → UDP syslog (unreliable) or TCP/TLS (reliable, preferred for AI)
Stage 3: Parse     → Regex for high-volume common types, LLM for novel/multi-vendor
Stage 4: Enrich    → CMDB + topology + change management context
Stage 5: Analyze   → Rate anomalies + LLM correlation → NOC alert
```

### Key Design Decisions

| Decision | Wrong approach | Right approach |
|----------|---------------|----------------|
| Transport | UDP syslog (drop on congestion) | TCP/TLS syslog (delivery guaranteed) |
| Parser | Regex-only (breaks on format changes) | Hybrid: regex + LLM for long tail |
| Timestamps | Device local time | Normalize to UTC at ingest |
| Deduplication | Aggressive (hides frequency) | Keep raw count — rate IS a signal |
| Enrichment | At query time | At ingest time — faster analysis |

### Why Rate Is a Signal

```
3 interface state changes  → investigate (maybe someone bumped a cable)
300 state changes/minute   → P1 (broadcast storm, STP loop, dying optic)
```
Both are the same syslog message type. Only the **rate** distinguishes them.
Always preserve frequency — don't deduplicate it away.

### Hybrid Parsing Economics

| Method | Speed | Cost | Use For |
|--------|-------|------|---------|
| Regex / Grok | ~1M/sec | ~$0 | OSPF, BGP, interface events (90%+ of volume) |
| LLM (Haiku) | ~100/sec | $0.0003/msg | Novel types, NX-OS, JunOS, multi-vendor |

### What's Next

- **Chapter 24**: Predictive Analytics — moving from reactive anomaly detection to proactive failure prediction using historical log patterns and time-series forecasting

> Logs are not archives of past disasters. They are a continuous real-time feed of your network's operational state — if you have a pipeline that reads them. This chapter built that pipeline.
